# 03 - Evaluación del Modelo

**Proyecto 04 — Clasificación y Extracción de Documentos con OCR + IA**

Métricas: Accuracy, F1-macro, Precision, Recall, Matriz de Confusión

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split

CATEGORIES = ['factura', 'recibo', 'contrato', 'constancia',
              'carta_formal', 'identificacion', 'otro']

# Cargar datos
texts, labels = [], []
for cat in CATEGORIES:
    for f in Path(f'../data/training/{cat}').glob('*.txt'):
        try:
            texts.append(f.read_text(encoding='utf-8'))
            labels.append(cat)
        except:
            pass

# Cargar modelo
model_data = joblib.load('../models/classifier_model.joblib')
model = model_data['model']
vectorizer = model_data['vectorizer']

_, X_test_raw, _, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
X_test = vectorizer.transform(X_test_raw)
y_pred = model.predict(X_test)

print('=== MÉTRICAS DE EVALUACIÓN ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1-macro: {f1_score(y_test, y_pred, average="macro"):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=CATEGORIES))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred, labels=CATEGORIES)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=ax
)
ax.set_title('Matriz de Confusión — LinearSVC + TF-IDF')
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
print('Matriz de confusión guardada: confusion_matrix.png')

## Conclusiones de Evaluación

- Algoritmo: TF-IDF (5000 features, ngram (1,2)) + LinearSVC (C=1.0)
- Validación cruzada 5-fold utilizada durante entrenamiento
- F1-macro es la métrica principal según el enunciado
- Matriz de confusión muestra qué tipos de documento se confunden entre sí